In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# =========================
# 프로젝트 경로 설정
# =========================

PROJECT_ROOT = Path("../..").resolve()
sys.path.append(str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PREPROCESSED_DIR = DATA_DIR / "preprocessed"
OUTPUT_DIR = DATA_DIR / "output"


print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("PREPROCESSED_DIR:", PREPROCESSED_DIR)


# src
from src.utils import read_csv_auto, check_dong_row_count

PROJECT_ROOT: C:\Users\user\Documents\GitHub\golmok-sesac_project
DATA_DIR: C:\Users\user\Documents\GitHub\golmok-sesac_project\data
PREPROCESSED_DIR: C:\Users\user\Documents\GitHub\golmok-sesac_project\data\preprocessed


In [2]:
# =========================
# CSV 파일 불러오기
# =========================

csv_files = list(PREPROCESSED_DIR.glob("*.csv"))

print(f"CSV 파일 개수: {len(csv_files)}")

for file in csv_files:
    print(file.name)

CSV 파일 개수: 10
cluster_df_yubeen.csv
merged_datamart.csv
merged_사업체.csv
merged_상권.csv
merged_인구.csv
merged_지하철.csv
동별 사업체수 현황.csv
서울시_마포구_동별_등록인구_전처리_최종.csv
서울시_상권분석_점포_입점_폐점율.csv
지하철 호선별 역별 승하차 인원 정보_2505~2605.csv


### 지하철 호선별 역별 승하차 인원 정보_2505~2605.csv

In [3]:
subway_path = PREPROCESSED_DIR / "지하철 호선별 역별 승하차 인원 정보_2505~2605.csv"

df_subway = read_csv_auto(subway_path)

df_subway.head()

로드 성공: 지하철 호선별 역별 승하차 인원 정보_2505~2605.csv / encoding=utf-8-sig / shape=(9504, 8)


,use_ymd,line_nm,stn_nm,ride_tot_psngr_co,alight_tot_psngr_co,reg_ymd,dong_nm,season_nm
0,20250501,2호선,신촌,33958,34606,20250504,대흥동,봄
1,20250501,2호선,아현,8748,8396,20250504,아현동,봄
2,20250501,2호선,이대,15331,16203,20250504,염리동,봄
3,20250501,2호선,합정,24507,26406,20250504,합정동,봄
4,20250501,2호선,홍대입구,72400,77967,20250504,서교동,봄


In [4]:
# =========================
# 지하철 데이터 전처리
# =========================

# 원본 데이터 복사
df_subway_clean = df_subway.copy()

# 전체 기간을 동별 기준으로 집계
df_subway_grouped = (
    df_subway_clean
    .groupby("dong_nm", as_index=False)
    .agg(
        # 동별 지하철 호선 수
        subway_line_count=("line_nm", "nunique"),

        # 동별 지하철 역 개수
        subway_station_count=("stn_nm", "nunique"),

        # 전체 기간 동별 총 승차 인원
        ride_tot_psngr_co=("ride_tot_psngr_co", "sum"),

        # 전체 기간 동별 총 하차 인원
        alight_tot_psngr_co=("alight_tot_psngr_co", "sum")
    )
)

# 승차량 + 하차량 = 전체 기간 지하철 총 승하차 인원 수
df_subway_grouped["subway_total_psngr_co"] = (
    df_subway_grouped["ride_tot_psngr_co"]
    + df_subway_grouped["alight_tot_psngr_co"]
)

# 승차량, 하차량 컬럼 삭제
df_subway_grouped = df_subway_grouped.drop(
    columns=["ride_tot_psngr_co", "alight_tot_psngr_co"]
)

# 결과 확인
print(df_subway_grouped.shape)
print(df_subway_grouped.columns)
df_subway_grouped.head()

(11, 4)
Index(['dong_nm', 'subway_line_count', 'subway_station_count',
       'subway_total_psngr_co'],
      dtype='str')


,dong_nm,subway_line_count,subway_station_count,subway_total_psngr_co
0,공덕동,4,1,30786905
1,대흥동,3,3,39710907
2,도화동,1,1,11063826
3,망원1동,1,1,13426972
4,상암동,3,1,23281286


In [5]:
save_path = PREPROCESSED_DIR / "merged_지하철.csv"

df_subway_grouped.to_csv(save_path, index=False, encoding="utf-8-sig")

save_path

WindowsPath('C:/Users/user/Documents/GitHub/golmok-sesac_project/data/preprocessed/merged_지하철.csv')

### 동별 사업체수 현황.csv
- 사업체 수 순위와 종사자 수 순위의 Spearman 상관계수
    - 0.798

In [6]:
business_path = PREPROCESSED_DIR / "동별 사업체수 현황.csv"

df_biz = read_csv_auto(business_path)

df_biz.head()

로드 성공: 동별 사업체수 현황.csv / encoding=utf-8-sig / shape=(17, 87)


,sido,sigungu,adm_dong,y2023_biz_cnt,y2023_emp_cnt,y2023_emp_male,y2023_emp_female,y2023_a_agri_forestry_fishing_biz,y2023_a_agri_forestry_fishing_emp,y2023_b_mining_biz,...,y2024_o_public_admin_biz,y2024_o_public_admin_emp,y2024_p_education_biz,y2024_p_education_emp,y2024_q_health_social_biz,y2024_q_health_social_emp,y2024_r_arts_sports_biz,y2024_r_arts_sports_emp,y2024_s_membership_repair_personal_biz,y2024_s_membership_repair_personal_emp
0,서울특별시,마포구,소계,55741,283983,153865,130118,6,15,0,...,63,6239,2635,16333,1375,14488,1691,4722,4088,10325
1,서울특별시,마포구,공덕동,2408,14508,7660,6848,0,0,0,...,4,532,152,684,124,1101,52,149,212,510
2,서울특별시,마포구,아현동,3601,21015,12252,8763,1,3,0,...,6,885,145,470,113,1283,63,140,265,795
3,서울특별시,마포구,도화동,2978,20809,11261,9548,1,3,0,...,2,204,136,601,107,995,70,148,211,670
4,서울특별시,마포구,용강동,3818,19374,10901,8473,1,1,0,...,6,417,166,841,82,983,92,218,233,724


In [7]:
df_biz_clean = df_biz.copy()

df_biz_clean = df_biz_clean[df_biz_clean["adm_dong"] != "소계"]
df_biz_clean.rename(columns={"adm_dong": "dong_nm"}, inplace=True)

keep_cols = (
    ["dong_nm"] +
    [col for col in df_biz_clean.columns if col.startswith("y2024_")]
)

df_biz_clean = df_biz_clean[keep_cols]

df_biz_clean.columns = [
    col.replace("y2024_", "") if col.startswith("y2024_") else col
    for col in df_biz_clean.columns
]

df_biz_clean = df_biz_clean.reset_index(drop=True)

# 4. 결과 확인
print(df_biz_clean.shape)
print(df_biz_clean.columns)
df_biz_clean.head()

(16, 43)
Index(['dong_nm', 'biz_cnt', 'emp_cnt', 'emp_male', 'emp_female',
       'a_agri_forestry_fishing_biz', 'a_agri_forestry_fishing_emp',
       'b_mining_biz', 'b_mining_emp', 'c_manufacturing_biz',
       'c_manufacturing_emp', 'd_electricity_gas_biz', 'd_electricity_gas_emp',
       'e_water_waste_biz', 'e_water_waste_emp', 'f_construction_biz',
       'f_construction_emp', 'g_wholesale_retail_biz',
       'g_wholesale_retail_emp', 'h_transport_storage_biz',
       'h_transport_storage_emp', 'i_accommodation_food_biz',
       'i_accommodation_food_emp', 'j_info_comm_biz', 'j_info_comm_emp',
       'k_finance_insurance_biz', 'k_finance_insurance_emp',
       'l_real_estate_biz', 'l_real_estate_emp', 'm_professional_sci_tech_biz',
       'm_professional_sci_tech_emp', 'n_business_support_biz',
       'n_business_support_emp', 'o_public_admin_biz', 'o_public_admin_emp',
       'p_education_biz', 'p_education_emp', 'q_health_social_biz',
       'q_health_social_emp', 'r_arts_sport

,dong_nm,biz_cnt,emp_cnt,emp_male,emp_female,a_agri_forestry_fishing_biz,a_agri_forestry_fishing_emp,b_mining_biz,b_mining_emp,c_manufacturing_biz,...,o_public_admin_biz,o_public_admin_emp,p_education_biz,p_education_emp,q_health_social_biz,q_health_social_emp,r_arts_sports_biz,r_arts_sports_emp,s_membership_repair_personal_biz,s_membership_repair_personal_emp
0,공덕동,2412,12769,6587,6182,1,1,0,0,79,...,4,532,152,684,124,1101,52,149,212,510
1,아현동,3593,20416,11616,8800,0,0,0,0,290,...,6,885,145,470,113,1283,63,140,265,795
2,도화동,2897,21761,11610,10151,2,4,0,0,48,...,2,204,136,601,107,995,70,148,211,670
3,용강동,3729,19150,10498,8652,0,0,0,0,63,...,6,417,166,841,82,983,92,218,233,724
4,대흥동,1988,9699,4577,5122,0,0,0,0,51,...,2,97,199,3466,101,912,59,288,182,559


In [8]:
save_path = PREPROCESSED_DIR / "merged_사업체.csv"

df_biz_clean.to_csv(save_path, index=False, encoding="utf-8-sig")

save_path

WindowsPath('C:/Users/user/Documents/GitHub/golmok-sesac_project/data/preprocessed/merged_사업체.csv')

### 서울시_마포구_동별_등록인구_전처리_최종.csv

In [9]:
pop_path = PREPROCESSED_DIR / "서울시_마포구_동별_등록인구_전처리_최종.csv"

df_pop = read_csv_auto(pop_path)

df_pop.head()

로드 성공: 서울시_마포구_동별_등록인구_전처리_최종.csv / encoding=utf-8-sig / shape=(16, 10)


,adstrd_cd_nm,pop_type,tot_pop,pop_00s,pop_10s,pop_20s,pop_30s,pop_40s,pop_50s,pop_60s_above
0,아현동,계,29566,2784,2319,2959,5940,5408,4281,5875
1,공덕동,계,36648,1710,2107,5308,6927,5047,5979,9570
2,도화동,계,20486,1153,1408,2404,3712,2854,3216,5739
3,용강동,계,21497,1725,2239,1964,3453,4191,3312,4613
4,대흥동,계,16221,500,1034,4855,2774,1913,2178,2967


In [10]:
df_pop = (
    df_pop
    .rename(columns={"adstrd_cd_nm": "dong_nm"})
    .drop(columns=["pop_type"])
)

print(df_pop.shape)
print(df_pop.columns)
df_pop.head()

(16, 9)
Index(['dong_nm', 'tot_pop', 'pop_00s', 'pop_10s', 'pop_20s', 'pop_30s',
       'pop_40s', 'pop_50s', 'pop_60s_above'],
      dtype='str')


,dong_nm,tot_pop,pop_00s,pop_10s,pop_20s,pop_30s,pop_40s,pop_50s,pop_60s_above
0,아현동,29566,2784,2319,2959,5940,5408,4281,5875
1,공덕동,36648,1710,2107,5308,6927,5047,5979,9570
2,도화동,20486,1153,1408,2404,3712,2854,3216,5739
3,용강동,21497,1725,2239,1964,3453,4191,3312,4613
4,대흥동,16221,500,1034,4855,2774,1913,2178,2967


In [11]:
save_path = PREPROCESSED_DIR / "merged_인구.csv"

df_pop.to_csv(save_path, index=False, encoding="utf-8-sig")

save_path

WindowsPath('C:/Users/user/Documents/GitHub/golmok-sesac_project/data/preprocessed/merged_인구.csv')

### 서울시_상권분석_점포_입점_폐점율.csv

In [12]:
input_path = PREPROCESSED_DIR / "서울시_상권분석_점포_입점_폐점율.csv"

store_raw = read_csv_auto(input_path)

store_raw.head()

로드 성공: 서울시_상권분석_점포_입점_폐점율.csv / encoding=utf-8-sig / shape=(45180, 15)


,stdr_yy_cd,stdr_qu_cd,signgu_cd,signgu_cd_nm,adstrd_cd,adstrd_cd_nm,trdar_cd,trdar_cd_nm,stor_co,similr_induty_stor_co,opbiz_rt,opbiz_stor_co,clsbiz_rt,clsbiz_stor_co,frc_stor_co
0,2023,1,11440,마포구,11440690,망원1동,3110540,망원나들목체육관,8,9,0,0,0,0,1
1,2023,1,11440,마포구,11440690,망원1동,3110540,망원나들목체육관,2,2,0,0,0,0,0
2,2023,1,11440,마포구,11440690,망원1동,3110540,망원나들목체육관,1,1,0,0,0,0,0
3,2023,1,11440,마포구,11440690,망원1동,3110540,망원나들목체육관,1,2,0,0,0,0,1
4,2023,1,11440,마포구,11440690,망원1동,3110540,망원나들목체육관,1,1,0,0,0,0,0


In [13]:
# =========================
# 2. 불필요한 코드/고정값 컬럼 삭제
# =========================
# 마포구만 사용하므로 signgu_cd, signgu_cd_nm 제거
# 행정동/상권은 코드 대신 이름 컬럼 사용

drop_cols = [
    "signgu_cd",
    "signgu_cd_nm",
    "adstrd_cd",
    "trdar_cd"
]

store_df = store_raw.drop(
    columns=drop_cols,
    errors="ignore"
).copy()

print("컬럼 삭제 후 shape:", store_df.shape)
display(store_df.head())


# =========================
# 3. 필수 컬럼 확인
# =========================

required_cols = [
    "stdr_yy_cd",
    "stdr_qu_cd",
    "adstrd_cd_nm",
    "trdar_cd_nm",
    "stor_co",
    "similr_induty_stor_co",
    "opbiz_rt",
    "opbiz_stor_co",
    "clsbiz_rt",
    "clsbiz_stor_co",
    "frc_stor_co"
]

missing_cols = [
    col for col in required_cols
    if col not in store_df.columns
]

if missing_cols:
    raise ValueError(f"필수 컬럼 누락: {missing_cols}")

print("필수 컬럼 확인 완료")


# =========================
# 4. 타입 정리
# =========================

num_cols = [
    "stdr_yy_cd",
    "stdr_qu_cd",
    "stor_co",
    "similr_induty_stor_co",
    "opbiz_rt",
    "opbiz_stor_co",
    "clsbiz_rt",
    "clsbiz_stor_co",
    "frc_stor_co"
]

for col in num_cols:
    store_df[col] = pd.to_numeric(
        store_df[col],
        errors="coerce"
    ).fillna(0)

store_df["stdr_yy_cd"] = store_df["stdr_yy_cd"].astype(int)
store_df["stdr_qu_cd"] = store_df["stdr_qu_cd"].astype(int)

display(store_df.head())


# =========================
# 5. 같은 상권 반복 여부 확인
# =========================

dup_count = store_df.duplicated(
    subset=[
        "stdr_yy_cd",
        "stdr_qu_cd",
        "adstrd_cd_nm",
        "trdar_cd_nm"
    ]
).sum()

print("같은 연도/분기/행정동/상권명 기준 중복 행 수:", dup_count)
print("업종 컬럼이 없으므로 중복 행은 상권 전체 현황으로 합산 처리합니다.")


# =========================
# 6. 상권 단위 집계
# =========================
# 같은 연도/분기/행정동/상권명 기준으로 합산
# 비율 컬럼(opbiz_rt, clsbiz_rt)은 합산하면 안 되므로 평균 사용

store_trdar_df = (
    store_df
    .groupby(
        [
            "stdr_yy_cd",
            "stdr_qu_cd",
            "adstrd_cd_nm",
            "trdar_cd_nm"
        ],
        as_index=False
    )
    .agg(
        stor_co=("stor_co", "sum"),
        similr_induty_stor_co=("similr_induty_stor_co", "sum"),
        opbiz_stor_co=("opbiz_stor_co", "sum"),
        clsbiz_stor_co=("clsbiz_stor_co", "sum"),
        frc_stor_co=("frc_stor_co", "sum"),
        opbiz_rt=("opbiz_rt", "mean"),
        clsbiz_rt=("clsbiz_rt", "mean")
    )
)

print("상권 단위 집계 후 shape:", store_trdar_df.shape)
display(store_trdar_df.head())


# =========================
# 7. 상권 단위 파생변수 생성
# =========================

denom = store_trdar_df["stor_co"].replace(0, np.nan)

store_trdar_df["similar_store_ratio"] = (
    store_trdar_df["similr_induty_stor_co"] / denom
)

store_trdar_df["franchise_ratio"] = (
    store_trdar_df["frc_stor_co"] / denom
)

store_trdar_df["open_store_ratio"] = (
    store_trdar_df["opbiz_stor_co"] / denom
)

store_trdar_df["close_store_ratio"] = (
    store_trdar_df["clsbiz_stor_co"] / denom
)

store_trdar_df["net_open_store_co"] = (
    store_trdar_df["opbiz_stor_co"]
    - store_trdar_df["clsbiz_stor_co"]
)

store_trdar_df["net_open_store_ratio"] = (
    store_trdar_df["net_open_store_co"] / denom
)

store_trdar_df["turnover_rate"] = (
    store_trdar_df["opbiz_rt"]
    + store_trdar_df["clsbiz_rt"]
)

store_trdar_df = store_trdar_df.fillna(0)

print("상권 단위 파생변수 생성 완료")
display(store_trdar_df.head())


# =========================
# 8. 행정동 단위 집계
# =========================
# 모델 병합 단위가 행정동이므로 행정동 단위로 다시 집계

store_dong_df = (
    store_trdar_df
    .groupby(
        [
            "stdr_yy_cd",
            "stdr_qu_cd",
            "adstrd_cd_nm"
        ],
        as_index=False
    )
    .agg(
        commercial_area_count=("trdar_cd_nm", "nunique"),

        stor_co=("stor_co", "sum"),
        similr_induty_stor_co=("similr_induty_stor_co", "sum"),
        opbiz_stor_co=("opbiz_stor_co", "sum"),
        clsbiz_stor_co=("clsbiz_stor_co", "sum"),
        frc_stor_co=("frc_stor_co", "sum"),

        opbiz_rt=("opbiz_rt", "mean"),
        clsbiz_rt=("clsbiz_rt", "mean"),

        avg_similar_store_ratio=("similar_store_ratio", "mean"),
        avg_franchise_ratio=("franchise_ratio", "mean"),
        avg_open_store_ratio=("open_store_ratio", "mean"),
        avg_close_store_ratio=("close_store_ratio", "mean"),
        avg_net_open_store_ratio=("net_open_store_ratio", "mean"),
        avg_turnover_rate=("turnover_rate", "mean")
    )
)

print("행정동 단위 집계 후 shape:", store_dong_df.shape)
display(store_dong_df.head())


# =========================
# 9. 행정동 전체 기준 비율 재계산
# =========================
# 상권별 평균 비율과 별도로,
# 행정동 전체 점포 수 기준 비율을 다시 계산

dong_denom = store_dong_df["stor_co"].replace(0, np.nan)

store_dong_df["similar_store_ratio_total"] = (
    store_dong_df["similr_induty_stor_co"] / dong_denom
)

store_dong_df["franchise_ratio_total"] = (
    store_dong_df["frc_stor_co"] / dong_denom
)

store_dong_df["open_store_ratio_total"] = (
    store_dong_df["opbiz_stor_co"] / dong_denom
)

store_dong_df["close_store_ratio_total"] = (
    store_dong_df["clsbiz_stor_co"] / dong_denom
)

store_dong_df["net_open_store_co"] = (
    store_dong_df["opbiz_stor_co"]
    - store_dong_df["clsbiz_stor_co"]
)

store_dong_df["net_open_store_ratio_total"] = (
    store_dong_df["net_open_store_co"] / dong_denom
)

store_dong_df["turnover_rate_total"] = (
    store_dong_df["open_store_ratio_total"]
    + store_dong_df["close_store_ratio_total"]
)

store_dong_df = store_dong_df.fillna(0)

print("행정동 단위 파생변수 생성 완료")
display(store_dong_df.head())


# =========================
# 10. 최신 연도/분기 데이터 추출
# =========================

latest_year = store_dong_df["stdr_yy_cd"].max()

latest_quarter = (
    store_dong_df
    .loc[store_dong_df["stdr_yy_cd"] == latest_year, "stdr_qu_cd"]
    .max()
)

store_dong_latest_df = store_dong_df[
    (store_dong_df["stdr_yy_cd"] == latest_year)
    & (store_dong_df["stdr_qu_cd"] == latest_quarter)
].copy()

print("최신 연도:", latest_year)
print("최신 분기:", latest_quarter)
print("최신 행정동 데이터 shape:", store_dong_latest_df.shape)

display(store_dong_latest_df.head())


# =========================
# 11. 모델용 컬럼 정리
# =========================

store_dong_latest_df = store_dong_latest_df[
    [
        "adstrd_cd_nm",

        "commercial_area_count",

        "stor_co",
        "similr_induty_stor_co",
        "frc_stor_co",

        "opbiz_stor_co",
        "clsbiz_stor_co",
        "net_open_store_co",

        "similar_store_ratio_total",
        "franchise_ratio_total",
        "open_store_ratio_total",
        "close_store_ratio_total",
        "net_open_store_ratio_total",
        "turnover_rate_total",

        "opbiz_rt",
        "clsbiz_rt",
        "avg_turnover_rate"
    ]
].copy()

store_dong_latest_df = store_dong_latest_df.rename(
    columns={
        "adstrd_cd_nm": "dong_nm",

        "commercial_area_count": "commercial_area_count",

        "stor_co": "store_count",
        "similr_induty_stor_co": "similar_store_count",
        "frc_stor_co": "franchise_store_count",

        "opbiz_stor_co": "open_store_count",
        "clsbiz_stor_co": "close_store_count",
        "net_open_store_co": "net_open_store_count",

        "similar_store_ratio_total": "similar_store_ratio",
        "franchise_ratio_total": "franchise_ratio",
        "open_store_ratio_total": "open_store_ratio",
        "close_store_ratio_total": "close_store_ratio",
        "net_open_store_ratio_total": "net_open_store_ratio",
        "turnover_rate_total": "store_turnover_ratio",

        "opbiz_rt": "avg_open_rate",
        "clsbiz_rt": "avg_close_rate"
    }
)

print("모델용 최신 행정동 데이터")
print(store_dong_latest_df.shape)
display(store_dong_latest_df.head())

컬럼 삭제 후 shape: (45180, 11)


,stdr_yy_cd,stdr_qu_cd,adstrd_cd_nm,trdar_cd_nm,stor_co,similr_induty_stor_co,opbiz_rt,opbiz_stor_co,clsbiz_rt,clsbiz_stor_co,frc_stor_co
0,2023,1,망원1동,망원나들목체육관,8,9,0,0,0,0,1
1,2023,1,망원1동,망원나들목체육관,2,2,0,0,0,0,0
2,2023,1,망원1동,망원나들목체육관,1,1,0,0,0,0,0
3,2023,1,망원1동,망원나들목체육관,1,2,0,0,0,0,1
4,2023,1,망원1동,망원나들목체육관,1,1,0,0,0,0,0


필수 컬럼 확인 완료


,stdr_yy_cd,stdr_qu_cd,adstrd_cd_nm,trdar_cd_nm,stor_co,similr_induty_stor_co,opbiz_rt,opbiz_stor_co,clsbiz_rt,clsbiz_stor_co,frc_stor_co
0,2023,1,망원1동,망원나들목체육관,8,9,0,0,0,0,1
1,2023,1,망원1동,망원나들목체육관,2,2,0,0,0,0,0
2,2023,1,망원1동,망원나들목체육관,1,1,0,0,0,0,0
3,2023,1,망원1동,망원나들목체육관,1,2,0,0,0,0,1
4,2023,1,망원1동,망원나들목체육관,1,1,0,0,0,0,0


같은 연도/분기/행정동/상권명 기준 중복 행 수: 44256
업종 컬럼이 없으므로 중복 행은 상권 전체 현황으로 합산 처리합니다.
상권 단위 집계 후 shape: (924, 11)


...

,stdr_yy_cd,stdr_qu_cd,adstrd_cd_nm,commercial_area_count,stor_co,similr_induty_stor_co,opbiz_stor_co,clsbiz_stor_co,frc_stor_co,opbiz_rt,...,avg_close_store_ratio,avg_net_open_store_ratio,avg_turnover_rate,similar_store_ratio_total,franchise_ratio_total,open_store_ratio_total,close_store_ratio_total,net_open_store_co,net_open_store_ratio_total,turnover_rate_total
0,2023,1,공덕동,7,909,979,24,19,70,2.502516,...,0.021503,0.008646,3.825515,1.077008,0.077008,0.026403,0.020902,5,0.005501,0.047305
1,2023,1,대흥동,5,477,523,22,14,46,4.441685,...,0.031706,0.028798,6.886411,1.096436,0.096436,0.046122,0.029350,8,0.016771,0.075472
2,2023,1,도화동,3,1502,1706,53,45,204,5.159314,...,0.034033,0.005635,8.790196,1.135819,0.135819,0.035286,0.029960,8,0.005326,0.065246
3,2023,1,망원1동,4,1713,1834,70,63,121,3.586098,...,0.029598,0.018142,5.797707,1.070636,0.070636,0.040864,0.036778,7,0.004086,0.077642
4,2023,1,망원2동,4,983,1068,46,45,85,3.485662,...,0.043463,0.000522,9.364931,1.086470,0.086470,0.046796,0.045778,1,0.001017,0.092574


최신 연도: 2025
최신 분기: 4
최신 행정동 데이터 shape: (16, 24)


,stdr_yy_cd,stdr_qu_cd,adstrd_cd_nm,commercial_area_count,stor_co,similr_induty_stor_co,opbiz_stor_co,clsbiz_stor_co,frc_stor_co,opbiz_rt,...,avg_close_store_ratio,avg_net_open_store_ratio,avg_turnover_rate,similar_store_ratio_total,franchise_ratio_total,open_store_ratio_total,close_store_ratio_total,net_open_store_co,net_open_store_ratio_total,turnover_rate_total
176,2025,4,공덕동,7,900,964,13,21,64,1.688827,...,0.022668,-0.005003,3.179674,1.071111,0.071111,0.014444,0.023333,-8,-0.008889,0.037778
177,2025,4,대흥동,5,507,542,17,21,35,3.080978,...,0.043006,-0.004548,5.670793,1.069034,0.069034,0.033531,0.041420,-4,-0.007890,0.074951
178,2025,4,도화동,3,1470,1655,27,34,185,1.561374,...,0.031914,-0.011149,4.076722,1.125850,0.125850,0.018367,0.023129,-7,-0.004762,0.041497
179,2025,4,망원1동,4,1716,1824,63,48,108,2.191916,...,0.033509,0.001803,5.145370,1.062937,0.062937,0.036713,0.027972,15,0.008741,0.064685
180,2025,4,망원2동,4,966,1042,31,25,76,2.182475,...,0.025972,0.006488,3.760866,1.078675,0.078675,0.032091,0.025880,6,0.006211,0.057971


모델용 최신 행정동 데이터
(16, 17)


,dong_nm,commercial_area_count,store_count,similar_store_count,franchise_store_count,open_store_count,close_store_count,net_open_store_count,similar_store_ratio,franchise_ratio,open_store_ratio,close_store_ratio,net_open_store_ratio,store_turnover_ratio,avg_open_rate,avg_close_rate,avg_turnover_rate
176,공덕동,7,900,964,64,13,21,-8,1.071111,0.071111,0.014444,0.023333,-0.008889,0.037778,1.688827,1.490847,3.179674
177,대흥동,5,507,542,35,17,21,-4,1.069034,0.069034,0.033531,0.041420,-0.007890,0.074951,3.080978,2.589815,5.670793
178,도화동,3,1470,1655,185,27,34,-7,1.125850,0.125850,0.018367,0.023129,-0.004762,0.041497,1.561374,2.515347,4.076722
179,망원1동,4,1716,1824,108,63,48,15,1.062937,0.062937,0.036713,0.027972,0.008741,0.064685,2.191916,2.953454,5.145370
180,망원2동,4,966,1042,76,31,25,6,1.078675,0.078675,0.032091,0.025880,0.006211,0.057971,2.182475,1.578391,3.760866


In [14]:

# =========================
# 12. 저장
# =========================
'''
store_trdar_output_path = (
    PREPROCESSED_DIR / "서울시_상권분석_점포_상권단위_전체현황.csv"
)

store_dong_output_path = (
    PREPROCESSED_DIR / "서울시_상권분석_점포_행정동단위_전체분기.csv"
)
'''
store_dong_latest_output_path = (
    PREPROCESSED_DIR / "merged_상권.csv"
)
'''
store_trdar_df.to_csv(
    store_trdar_output_path,
    index=False,
    encoding="utf-8-sig"
)

store_dong_df.to_csv(
    store_dong_output_path,
    index=False,
    encoding="utf-8-sig"
)
'''
store_dong_latest_df.to_csv(
    store_dong_latest_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
'''
print("상권 단위 전체현황:", store_trdar_output_path)
print("행정동 단위 전체분기:", store_dong_output_path)
'''
print("행정동 단위 최신분기:", store_dong_latest_output_path)

print(store_dong_latest_df.shape)
print(store_dong_latest_df.columns)
print(store_dong_latest_df.head())

저장 완료
행정동 단위 최신분기: C:\Users\user\Documents\GitHub\golmok-sesac_project\data\preprocessed\merged_상권.csv
(16, 17)
Index(['dong_nm', 'commercial_area_count', 'store_count',
       'similar_store_count', 'franchise_store_count', 'open_store_count',
       'close_store_count', 'net_open_store_count', 'similar_store_ratio',
       'franchise_ratio', 'open_store_ratio', 'close_store_ratio',
       'net_open_store_ratio', 'store_turnover_ratio', 'avg_open_rate',
       'avg_close_rate', 'avg_turnover_rate'],
      dtype='str')
    dong_nm  commercial_area_count  store_count  similar_store_count  \
176     공덕동                      7          900                  964   
177     대흥동                      5          507                  542   
178     도화동                      3         1470                 1655   
179    망원1동                      4         1716                 1824   
180    망원2동                      4          966                 1042   

     franchise_store_count  open_store_cou